In [2]:

# %%
import anndata as ad
import h5py
import json
import numpy as np
import pandas as pd
from pathlib import Path
import os
import ijson
import scanpy as sc
import anndata as ad
import scipy.sparse as sp


# %% [markdown]
# %%
from cell_type_mapper.cli.from_specified_markers import FromSpecifiedMarkersRunner

from cell_type_mapper.cli.map_to_on_the_fly_markers import OnTheFlyMapper

from cell_type_mapper.cli.precompute_stats_scrattch import PrecomputationScrattchRunner
from cell_type_mapper.cli.reference_markers import ReferenceMarkerRunner
from cell_type_mapper.cli.query_markers import QueryMarkerRunner




In [3]:

HOME = Path.home()

ROOT = HOME / "Projects/ASAP/cell_type_mapper"


CHUNK_SIZE = 40000
N_RUNNERS_UP = 5
RNG_SEED = 11235813
N_PROCESSORS = 6
MAX_GB = 32.0


In [4]:

# In[ ]:
## e.g. for xylena data
XYLENA2_FULL = "xyl2_full.h5ad"

XYLENA2_TRAIN = "xyl2_train.h5ad"
XYLENA2_TEST = "xyl2_test.h5ad"
XYLENA2_QUERY_A = "xyl2_query.h5ad"
XYLENA2_QUERY_B = "xyl2_queryB.h5ad"


# linux
XYLENA2_PATH = "scdata/xylena"




## Make our ATLAS_DIR
ATLAS_NAME = "XYLENA2"
ATLAS_DIR = ROOT / ATLAS_NAME


# In[ ]:

# mac

# linux
data_path = ROOT / XYLENA2_PATH

In [5]:
if False:

    # mac

    # linux
    data_path = ROOT / XYLENA2_PATH
    # In[ ]: Load raw data
    ########################
    # 0. LOAD TRAIN DATA
    ########################

    train_filen = data_path / XYLENA2_TRAIN
    # train_ad = ad.read_h5ad(train_filen, backed="r")
    train_ad = ad.read_h5ad(train_filen)
    # # In[ ]:  make test anndata

    # test_filen = data_path / XYLENA2_TEST
    # test_ad = ad.read_h5ad(test_filen, backed="r")
    # queryA_filen = data_path / XYLENA2_QUERY_A
    # query_ad = ad.read_h5ad(queryA_filen, backed="r")
    # queryB_filen = data_path / XYLENA2_QUERY_B
    # queryB_ad = ad.read_h5ad(queryB_filen, backed="r")
    # %%  MAKE a local copy of the training data which is encoded with ensemble_ids


    train_ad.var["gene_name"] = train_ad.var.index
    train_ad.var_names = train_ad.var["gene_ids"]

    # also coudl use lbl8r.model.utils._data.sparsify_adata

    train_ad.X = sp.csr_matrix(train_ad.X)

    # TODO: make recoded copies of these...
    # test_filen = data_path / XYLENA2_TEST
    # test_ad = ad.read_h5ad(test_filen, backed="r")
    # queryA_filen = data_path / XYLENA2_QUERY_A
    # query_ad = ad.read_h5ad(queryA_filen, backed="r")
    # queryB_filen = data_path / XYLENA2_QUERY_B
    # queryB_ad = ad.read_h5ad(queryB_filen, backed="r")



    # In[ ]:  Make taxonomy

    def class_assign(x):
        if x in ['ExN', 'InN']:
            return 'neuron'
        elif x in ['Oligo', 'VC', 'OPC', 'Astro', 'MG']:
            return 'non-neuronal'



    # mapping = {
    #     "oligo": "Oligo",
    #     "opc": "OPC",
    #     "glutamatergic": "ExN",
    #     "gabergic": "InN",
    #     "astrocyte": "Astro",
    #     "immune": "MG",
    #     "blood": "VC",
    # }

    subc_mapping = {
        "Oligo": "oligo",
        "OPC": "opc",
        "ExN": "glutamatergic",
        "InN": "gabaergic",
        "Astro": "astrocyte",
        "MG": "immune",
        "VC": "blood_vessel",
    }


    train_ad.obs['class'] = train_ad.obs['cell_type'].apply(class_assign)
    train_ad.obs['subclass'] = train_ad.obs['cell_type'].map(subc_mapping)


heirarchy = ["class", "subclass"]


In [6]:

if not ATLAS_DIR.exists():
    ATLAS_DIR.mkdir()

atlas_filenm = ATLAS_DIR / f"{ATLAS_NAME}_atlas_adata.h5ad"
# train_ad = ad.read_h5ad(atlas_filenm)


In [7]:

heirarchy = ["class", "subclass"]

stats_path = ATLAS_DIR / f"{ATLAS_NAME}_precomputed_stats.h5"

precomputation_config = {
    'hierarchy': heirarchy,
    'h5ad_path': f"{atlas_filenm}",
    'output_path': f"{stats_path}",
    'n_processors': N_PROCESSORS,
    'normalization': 'raw',
}

# # %%
# !python -m cell_type_mapper.cli.precompute_stats_scrattch \
# --hierarchy f"{heirarchy}" \
# --h5ad_path f"{atlas_filenm}"\
# --n_processors f"{N_PROCESSORS}" \
# --normalization raw \
# --clobber True \
# --output_path f"{stats_path}" 

In [12]:

# the args=[] is important to prevent the PrecomputationScrattchRunner from grabbing
# any command line arguments when you invoke your python script
precomputation_runner = PrecomputationScrattchRunner(
    args=[], input_data=precomputation_config)
# %%
precomputation_runner.run()

print('\n====done with precomputed_stats====\n')



finally process 4623 tot 3.24e+02 reading 1.36e+01 writing 1.42e-02
finally process 4531 tot 3.69e+02 reading 1.34e+01 writing 2.02e-02
finally process 4473 tot 3.93e+02 reading 1.31e+01 writing 1.73e-02
finally process 4580 tot 3.94e+02 reading 1.53e+01 writing 7.10e-03
finally process 4491 tot 3.99e+02 reading 1.57e+01 writing 6.92e-03
finally process 4434 tot 4.03e+02 reading 1.38e+01 writing 1.28e-02

====done with precomputed_stats====



In [13]:

# %% REFERENCE MARKERS
# # %%
# !python -m cell_type_mapper.cli.reference_markers \
# --precomputed_path_list "['pipeline_example_data/precomputed_stats.h5']" \
# --n_valid 20 \
# --n_processors 3 \
# --output_dir pipeline_example_data \
# --clobber True
# reference_config = {
#     "precomputed_path_list": ["pipeline_example_data/precomputed_stats.h5"],
#     "n_valid": 20,
#     "output_dir": "pipeline_example_data",
#     "clobber": True,
# }

In [14]:

output_dir = ATLAS_DIR / "reference"
REFERENCE_DIR = ATLAS_DIR / "reference"
reference_config = {
    'precomputed_path_list': [f"{stats_path}"],
    'n_valid': 20,
    'n_processors': N_PROCESSORS,
    'output_dir': f"{output_dir}",
    'clobber': True
}

reference_runner = ReferenceMarkerRunner(
    args=[], input_data=reference_config)
# %%
reference_runner.run()
print('\n====done with reference_markers====\n')

# one thread 2 seconds
# 6 threads ?? seconds

JAH::::  0 create_input_to_output_map
JAH::::  1 read_df_from_h5ad
writing /Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/reference/reference_markers.h5
JAH::::  2 TaxonomyTree precomputed_path='/Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/XYLENA2_precomputed_stats.h5'
Starting XYLENA2_precomputed_stats.h5
16 of 21 taxon pairs in 1.78e+00 sec; predict 5.55e-01 sec of 2.33e+00 sec left
24 of 21 taxon pairs in 1.87e+00 sec; predict -2.33e-01 sec of 1.63e+00 sec left
Initial marker discovery took 1.89e+00 seconds
joining took 3.834963e-03 seconds
joining took 4.865885e-03 seconds
Transposing markers took 3.89e-01 seconds
Copying to /Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/reference/reference_markers.h5 took 4.75e-04 seconds
Wrote reference_markers.h5
JAH::::  3 find_markers_for_all_taxonomy_pairs precomputed_path='/Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/XYLENA2_precomputed_stats.h5'
completed in 2.61e+00 seconds
JAH::::  4 completed in 2.61e+00 s

In [16]:
# %%  QUERY MARKERS

query_markers_path = ATLAS_DIR / f"reference/query_markers.json"
reference_markers_path = ATLAS_DIR / f"reference/reference_markers.h5"

query_config = {
    'output_path': f"{query_markers_path}",
    'reference_marker_path_list': [f"{reference_markers_path}"],
    'n_per_utility': 10,
    'n_processors': N_PROCESSORS,
}

query_runner = QueryMarkerRunner(
    args=[], input_data=query_config)

# %%
query_runner.run()
print('\n====done with query_markers====\n')


QUERY MARKER FINDER RAN SUCCESSFULLY in 2.28e+00 seconds

====done with query_markers====



In [38]:

# mac

# linux
data_path = ROOT / XYLENA2_PATH
########################
# 0.1 PREP TRAIN/TEST DATA
########################

# test_filen = data_path / XYLENA2_TEST
# test_ad = ad.read_h5ad(test_filen)
# # test_ad = ad.read_h5ad(test_filen, backed="r")

queryA_filen = data_path / XYLENA2_QUERY_A
# query_ad = ad.read_h5ad(queryA_filen, backed="r")
query_ad = ad.read_h5ad(queryA_filen)

queryB_filen = data_path / XYLENA2_QUERY_B
# queryB_ad = ad.read_h5ad(queryB_filen, backed="r")
queryB_ad = ad.read_h5ad(queryB_filen)
# %%  MAKE a local copy of the training data which is encoded with ensemble_ids


def prep_adata_for_MMC(adata: ad.AnnData) -> ad.AnnData:

    adata.var["gene_name"] = adata.var.index

    adata.var_names = adata.var["gene_ids"]
    adata.X = sp.csr_matrix(adata.X)
    return adata


# test_ad = prep_adata_for_MMC(test_ad)
# test_filenm = ATLAS_DIR / f"{ATLAS_NAME}_test_adata.h5ad"
# test_ad.write_h5ad(test_filenm)


query_ad = prep_adata_for_MMC(query_ad)
query_filenm = ATLAS_DIR / f"{ATLAS_NAME}_queryA_adata.h5ad"
query_ad.write_h5ad(query_filenm)

queryB_ad = prep_adata_for_MMC(queryB_ad)
queryB_filenm = ATLAS_DIR / f"{ATLAS_NAME}_queryB_adata.h5ad"
queryB_ad.write_h5ad(queryB_filenm) 


heirarchy = ["class", "subclass"]

In [40]:
query_ad.obs['cell_type'].value_counts()


Series([], Name: count, dtype: int64)

In [43]:

train_ad.obs['cell_type'].value_counts()

cell_type
Oligo    360708
ExN      117025
InN       91702
Astro     74514
MG        57661
OPC       52388
VC         9898
Name: count, dtype: int64

In [44]:

test_ad.obs['cell_type'].value_counts()

cell_type
Oligo    160846
ExN       37485
Astro     33438
InN       29584
MG        23185
OPC       20534
VC         3444
Name: count, dtype: int64

In [12]:
test_filenm.stem

'XYLENA2_test_adata'

In [14]:
### now map test data into the reference space...


TMP_DIR = ROOT / "tmp"
if not TMP_DIR.exists():
    TMP_DIR.mkdir()


CHUNK_SIZE = 40000
N_RUNNERS_UP = 5
RNG_SEED = 11235813
N_PROCESSORS = 8
MAX_GB = 48.0

# %%  force CPU
os.environ["AIBS_BKP_USE_TORCH"] = "false"

os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

#
FILE_ROOT = test_filenm.stem
REFERENCE = ATLAS_NAME
DATE = "20250208"

EXTENDED_RESULTS = f"{FILE_ROOT}.mmc.{REFERENCE}.{DATE}.json"
LOG_FILE = f"{FILE_ROOT}.mmc.{REFERENCE}_log.{DATE}.txt"
CSV_RESULTS = f"{FILE_ROOT}.mmc.{REFERENCE}_results.{DATE}.csv"


RESULTS_DIR = ATLAS_DIR / f"MMC.{REFERENCE}_RESULTS"
if not RESULTS_DIR.exists():
    RESULTS_DIR.mkdir()


config = {
    # "query_path": f"{ASAP_DATA}/{FILE_ROOT}.h5ad",
    "query_path": f"{test_filenm}",
    "tmp_dir": f"{TMP_DIR}",
    "extended_result_path": f"{RESULTS_DIR / EXTENDED_RESULTS}",
    "csv_result_path": f"{RESULTS_DIR / CSV_RESULTS}",
    "log_path": f"{RESULTS_DIR / LOG_FILE}",
    "cloud_safe": False,
    "verbose_csv": True,
    "n_processors": N_PROCESSORS,
    "max_gb": MAX_GB,
    "map_to_ensembl": True,
    "type_assignment": {
        "normalization": "raw",
        "bootstrap_iteration": 100,
        "bootstrap_factor": 0.5,
        "chunk_size": CHUNK_SIZE,
        "n_runners_up": N_RUNNERS_UP,
        "rng_seed": RNG_SEED,
    },
    "precomputed_stats": {"path": f"{stats_path}"},
    "reference_markers": {"log2_fold_min_th": 0.5},
    "query_markers": {"n_per_utility": 15, "genes_at_a_time": 1},
}



In [15]:

#%%
runner = OnTheFlyMapper(args=[], input_data=config)
runner.run()





starting to find reference markers
JAH::::  0 create_input_to_output_map
JAH::::  1 read_df_from_h5ad
writing /Users/ergonyc/Projects/ASAP/cell_type_mapper/tmp/tmp32_g1nto/tmpoqv1wqpv/reference_markers.h5
JAH::::  2 TaxonomyTree precomputed_path='/Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/XYLENA2_precomputed_stats.h5'
Starting XYLENA2_precomputed_stats.h5
16 of 21 taxon pairs in 1.62e+00 sec; predict 5.05e-01 sec of 2.12e+00 sec left
24 of 21 taxon pairs in 1.70e+00 sec; predict -2.13e-01 sec of 1.49e+00 sec left
Initial marker discovery took 1.74e+00 seconds
joining took 4.328251e-03 seconds
joining took 5.668163e-03 seconds
Transposing markers took 4.41e-01 seconds
Copying to /Users/ergonyc/Projects/ASAP/cell_type_mapper/tmp/tmp32_g1nto/tmpoqv1wqpv/reference_markers.h5 took 1.96e-04 seconds
Wrote reference_markers.h5
JAH::::  3 find_markers_for_all_taxonomy_pairs precomputed_path='/Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/XYLENA2_precomputed_stats.h5'
completed

/Users/ergonyc/Projects/ASAP/cell_type_mapper/src/cell_type_mapper/taxonomy/utils.py:245: UserWarning: This taxonomy has no mapping from leaf_node -> rows in the cell by gene matrix
  warnings.warn("This taxonomy has no mapping from leaf_node -> rows "


BENCHMARK: spent 7.4407e-02 seconds creating query marker cache
Running CPU implementation of type assignment.
BENCHMARK: spent 4.0007e+02 seconds assigning cell types
Writing marker genes to output file
FILE TRACKER: cleaning up ../file_tracker_wxzxfsc3
MAPPING FROM SPECIFIED MARKERS RAN SUCCESSFULLY
CLEANING UP
MAPPING FROM ON-THE-FLY MARKERS RAN SUCCESSFULLY


In [16]:




### NOw lets look at the Test data to see how we did.

# load test_ad.obs 
obs = test_ad.obs.copy()


In [31]:

# load resuults.
# results header is the first 3 lines
results = pd.read_csv( (RESULTS_DIR / CSV_RESULTS), header=3)
results.set_index( "cell_id", inplace=True)
results.index.name = None

long_to_short_mapping = {
    "oligo": "Oligo",
    "opc": "OPC",
    "glutamatergic": "ExN",
    "gabergic": "InN",
    "astrocyte": "Astro",
    "immune": "MG",
    "blood": "VC",
}
results['cell_type_mapped'] = results['subclass_label'].map(long_to_short_mapping)


results.head()




,class_label,class_name,class_bootstrapping_probability,class_correlation_coefficient,class_aggregate_probability,subclass_label,subclass_name,subclass_alias,subclass_bootstrapping_probability,subclass_correlation_coefficient,subclass_aggregate_probability,cell_type_mapped
ACCAATATCACGTTAA-1_UMARY-4638-ARC,neuron,neuron,1.00,0.8183,1.00,glutamatergic,glutamatergic,glutamatergic,1.00,0.5941,1.00,ExN
GACTTACAGGCAATAG-1_UMARY-4638-ARC,neuron,neuron,1.00,0.8332,1.00,glutamatergic,glutamatergic,glutamatergic,1.00,0.5843,1.00,ExN
TAAGTGCTCCACCTGT-1_UMARY-4638-ARC,neuron,neuron,1.00,0.7630,1.00,glutamatergic,glutamatergic,glutamatergic,0.68,0.1728,0.68,ExN
TATAGGTGTTTGGTTC-1_UMARY-4638-ARC,neuron,neuron,0.95,0.6837,0.95,glutamatergic,glutamatergic,glutamatergic,1.00,0.6147,0.95,ExN
GTCTTGCTCATAACGC-1_UMARY-4638-ARC,neuron,neuron,1.00,0.8142,1.00,glutamatergic,glutamatergic,glutamatergic,0.99,0.5041,0.99,ExN


In [ ]:
# merge and 

obs = obs.merge(results, left_index=True, right_index=True, how="left")



,cell_type,cell_type_mapped
ACCAATATCACGTTAA-1_UMARY-4638-ARC,ExN,ExN
GACTTACAGGCAATAG-1_UMARY-4638-ARC,ExN,ExN
TAAGTGCTCCACCTGT-1_UMARY-4638-ARC,ExN,ExN
TATAGGTGTTTGGTTC-1_UMARY-4638-ARC,ExN,ExN
GTCTTGCTCATAACGC-1_UMARY-4638-ARC,ExN,ExN


In [33]:

# make a confusion matrix
confusion = pd.crosstab(obs['cell_type'], obs['cell_type_mapped'])
confusion


cell_type_mapped,Astro,ExN,MG,OPC,Oligo
cell_type,,,,,
Astro,32944,14,6,68,365
ExN,181,35248,23,401,796
InN,24,14,4,137,65
MG,161,14,22819,36,135
OPC,7,4,2,20438,60
Oligo,11,37,4,16,160639
VC,58,7,241,12,74


In [24]:
results.head()

,class_label,class_name,class_bootstrapping_probability,class_correlation_coefficient,class_aggregate_probability,subclass_label,subclass_name,subclass_alias,subclass_bootstrapping_probability,subclass_correlation_coefficient,subclass_aggregate_probability
cell_id,,,,,,,,,,,
ACCAATATCACGTTAA-1_UMARY-4638-ARC,neuron,neuron,1.00,0.8183,1.00,glutamatergic,glutamatergic,glutamatergic,1.00,0.5941,1.00
GACTTACAGGCAATAG-1_UMARY-4638-ARC,neuron,neuron,1.00,0.8332,1.00,glutamatergic,glutamatergic,glutamatergic,1.00,0.5843,1.00
TAAGTGCTCCACCTGT-1_UMARY-4638-ARC,neuron,neuron,1.00,0.7630,1.00,glutamatergic,glutamatergic,glutamatergic,0.68,0.1728,0.68
TATAGGTGTTTGGTTC-1_UMARY-4638-ARC,neuron,neuron,0.95,0.6837,0.95,glutamatergic,glutamatergic,glutamatergic,1.00,0.6147,0.95
GTCTTGCTCATAACGC-1_UMARY-4638-ARC,neuron,neuron,1.00,0.8142,1.00,glutamatergic,glutamatergic,glutamatergic,0.99,0.5041,0.99


In [22]:
obs

,total_counts,total_counts_rb,pct_counts_rb,total_counts_mt,pct_counts_mt,doublet_score,batch,cohort,sample,n_genes_by_counts,...,train_sample,test_sample,queryA,queryB,cell_type,n_counts,n_genes,s_score,g2m_score,phase
ACCAATATCACGTTAA-1_UMARY-4638-ARC,51635.0,243.0,0.470611,55.0,0.106517,0.082297,batch4,nabec,UMARY-4638-ARC,8598,...,False,True,False,False,ExN,51635.0,8598,-0.035443,-0.019148,G1
GACTTACAGGCAATAG-1_UMARY-4638-ARC,51469.0,298.0,0.578989,23.0,0.044687,0.069975,batch4,nabec,UMARY-4638-ARC,8585,...,False,True,False,False,ExN,51469.0,8585,-0.041942,-0.063812,G1
TAAGTGCTCCACCTGT-1_UMARY-4638-ARC,50292.0,190.0,0.377794,53.0,0.105385,0.103540,batch4,nabec,UMARY-4638-ARC,8100,...,False,True,False,False,ExN,50292.0,8100,0.004580,-0.036680,S
TATAGGTGTTTGGTTC-1_UMARY-4638-ARC,50186.0,203.0,0.404495,64.0,0.127526,0.071860,batch4,nabec,UMARY-4638-ARC,8409,...,False,True,False,False,ExN,50186.0,8409,0.018357,-0.066770,S
GTCTTGCTCATAACGC-1_UMARY-4638-ARC,49037.0,136.0,0.277342,68.0,0.138671,0.103540,batch4,nabec,UMARY-4638-ARC,8475,...,False,True,False,False,ExN,49037.0,8475,-0.000117,-0.045870,G1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCCTGTGCAAAGCTCC-1_UMARY-933-ARC,503.0,0.0,0.000000,0.0,0.000000,0.004048,batch5,nabec,UMARY-933-ARC,397,...,False,True,False,False,MG,503.0,397,-0.027959,-0.029896,G1
TGGGCATGTTGCGGAT-1_UMARY-933-ARC,522.0,2.0,0.383142,0.0,0.000000,0.007181,batch5,nabec,UMARY-933-ARC,404,...,False,True,False,False,OPC,522.0,404,-0.032470,-0.024418,G1
ACTGAATGTTTCCGGC-1_UMARY-933-ARC,456.0,0.0,0.000000,0.0,0.000000,0.006835,batch5,nabec,UMARY-933-ARC,354,...,False,True,False,False,OPC,456.0,354,-0.025604,-0.031218,G1
AAGCGCTGTGCATTAG-1_UMARY-933-ARC,442.0,0.0,0.000000,0.0,0.000000,0.058472,batch5,nabec,UMARY-933-ARC,363,...,False,True,False,False,InN,442.0,363,0.059629,-0.035629,S


In [34]:
atlas_filenm

PosixPath('/Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/XYLENA2_atlas_adata.h5ad')

In [36]:
### now remap the train data into the reference space...

train_filenm = atlas_filenm
#
FILE_ROOT = train_filenm.stem
REFERENCE = ATLAS_NAME
DATE = "20250208"

EXTENDED_RESULTS = f"{FILE_ROOT}.mmc.{REFERENCE}.{DATE}.json"
LOG_FILE = f"{FILE_ROOT}.mmc.{REFERENCE}_log.{DATE}.txt"
CSV_RESULTS = f"{FILE_ROOT}.mmc.{REFERENCE}_results.{DATE}.csv"


RESULTS_DIR = ATLAS_DIR / f"MMC.{REFERENCE}_RESULTS"
if not RESULTS_DIR.exists():
    RESULTS_DIR.mkdir()


config = {
    # "query_path": f"{ASAP_DATA}/{FILE_ROOT}.h5ad",
    "query_path": f"{train_filenm}",
    "tmp_dir": f"{TMP_DIR}",
    "extended_result_path": f"{RESULTS_DIR / EXTENDED_RESULTS}",
    "csv_result_path": f"{RESULTS_DIR / CSV_RESULTS}",
    "log_path": f"{RESULTS_DIR / LOG_FILE}",
    "cloud_safe": False,
    "verbose_csv": True,
    "n_processors": N_PROCESSORS,
    "max_gb": MAX_GB,
    "map_to_ensembl": True,
    "type_assignment": {
        "normalization": "raw",
        "bootstrap_iteration": 100,
        "bootstrap_factor": 0.5,
        "chunk_size": CHUNK_SIZE,
        "n_runners_up": N_RUNNERS_UP,
        "rng_seed": RNG_SEED,
    },
    "precomputed_stats": {"path": f"{stats_path}"},
    "reference_markers": {"log2_fold_min_th": 0.5},
    "query_markers": {"n_per_utility": 15, "genes_at_a_time": 1},
}

#%%
runner = OnTheFlyMapper(args=[], input_data=config)
runner.run()



starting to find reference markers
JAH::::  0 create_input_to_output_map
JAH::::  1 read_df_from_h5ad
writing /Users/ergonyc/Projects/ASAP/cell_type_mapper/tmp/tmp3qf5j7u9/tmpxwlvf3mx/reference_markers.h5
JAH::::  2 TaxonomyTree precomputed_path='/Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/XYLENA2_precomputed_stats.h5'
Starting XYLENA2_precomputed_stats.h5
16 of 21 taxon pairs in 1.58e+00 sec; predict 4.94e-01 sec of 2.08e+00 sec left
24 of 21 taxon pairs in 1.67e+00 sec; predict -2.08e-01 sec of 1.46e+00 sec left
Initial marker discovery took 1.71e+00 seconds
joining took 5.136013e-03 seconds
joining took 4.549265e-03 seconds
Transposing markers took 5.27e-01 seconds
Copying to /Users/ergonyc/Projects/ASAP/cell_type_mapper/tmp/tmp3qf5j7u9/tmpxwlvf3mx/reference_markers.h5 took 1.87e-04 seconds
Wrote reference_markers.h5
JAH::::  3 find_markers_for_all_taxonomy_pairs precomputed_path='/Users/ergonyc/Projects/ASAP/cell_type_mapper/XYLENA2/XYLENA2_precomputed_stats.h5'
completed

/Users/ergonyc/Projects/ASAP/cell_type_mapper/src/cell_type_mapper/taxonomy/utils.py:245: UserWarning: This taxonomy has no mapping from leaf_node -> rows in the cell by gene matrix
  warnings.warn("This taxonomy has no mapping from leaf_node -> rows "


BENCHMARK: spent 8.0749e-02 seconds creating query marker cache
Running CPU implementation of type assignment.
560000 of 763896 cells in 4.04e+01 min; predict 1.47e+01 min of 5.51e+01 min left
BENCHMARK: spent 2.5978e+03 seconds assigning cell types
Writing marker genes to output file
FILE TRACKER: cleaning up ../file_tracker__nepfch7
MAPPING FROM SPECIFIED MARKERS RAN SUCCESSFULLY
CLEANING UP
MAPPING FROM ON-THE-FLY MARKERS RAN SUCCESSFULLY


In [37]:
### NOw lets look at the Test data to see how we did.

# load test_ad.obs 
train_ad = ad.read_h5ad(train_filenm)
obs = train_ad.obs.copy()


# load resuults.
# results header is the first 3 lines
results = pd.read_csv( (RESULTS_DIR / CSV_RESULTS), header=3)
results.set_index( "cell_id", inplace=True)
results.index.name = None

long_to_short_mapping = {
    "oligo": "Oligo",
    "opc": "OPC",
    "glutamatergic": "ExN",
    "gabergic": "InN",
    "astrocyte": "Astro",
    "immune": "MG",
    "blood": "VC",
}
results['cell_type_mapped'] = results['subclass_label'].map(long_to_short_mapping)



obs = obs.merge(results, left_index=True, right_index=True, how="left")


# make a confusion matrix
confusion = pd.crosstab(obs['cell_type'], obs['cell_type_mapped'])
confusion



cell_type_mapped,Astro,ExN,MG,OPC,Oligo
cell_type,,,,,
Astro,73496,38,14,119,721
ExN,197,113630,56,595,404
InN,93,60,11,395,137
MG,498,30,56739,88,268
OPC,63,12,3,51779,118
Oligo,17,40,7,17,360490
VC,148,15,472,33,145
